<div align="center">

# 🎡 Analítica de Clientes y Pipeline de Predicción de Churn
**Framework de Ciencia de Datos · Caso de Negocio — Parques AMVA 2024–2025**

</div>

---

| Campo | Detalle |
| :--- | :--- |
| **Autor** | Bernardo Adolfo Gómez Montoya |
| **Versión** | 1.0 |
| **Metodología** | Flujo de Trabajo Ágil — 7 Tarjetas Operacionales |
| **Volumen del Dataset** | 304,799 registros brutos · 221,004 post-ETL · 189,384 usuarios únicos |
| **Modelos Centrales** | Regresión Logística vs. Random Forest |
| **Features del modelo** | 12 features (RFM + demográficas + temporales + comportamentales) |

---

## 🔬 Bitácora de Auditoría y Tratamiento de Datos

| # | Hallazgo de auditoría | Tratamiento aplicado |
|:--|:-----------------------|:----------------------|
| 1 | `tarifa` con 415 variantes por casing/tildes/espacios internos | Normalización `unicodedata` + colapso de corrupciones → 5 categorías reales |
| 2 | `tipo_venta` con 346 variantes | Misma normalización → 2 categorías reales (INDIVIDUAL / EMPRESARIAL) |
| 3 | `edad` sin cota superior — usuarios con >100 años | `clip(lower=0, upper=100)` — sella ambos extremos |
| 4 | Señales comportamentales (fin de semana, diversidad temporal) no incorporadas al modelo | 12 features en total: + `pct_fin_semana`, `n_meses_activo`, one-hot de franja/edad/género |
| 5 | `genero` y `categoria_edad` calculados pero no usados como predictores | Incorporados vía `pd.get_dummies` (one-hot encoding real) |
| 6 | `grupo` normalizado pero no usado downstream | Documentado como columna de contexto, no feature del modelo |

---

## 📋 Tabla de Contenido

- [Tarjeta 1 — ETL Robusto y Auditoría de Calidad](#tarjeta-1)
- [Tarjeta 2 — Ingeniería de Características RFM Extendida](#tarjeta-2)
- [Tarjeta 3 — Segmentación K-Means](#tarjeta-3)
- [Tarjeta 4 — Modelado Supervisado (Churn)](#tarjeta-4)
- [Tarjeta 5 — Evaluación de Modelos y Curva ROC](#tarjeta-5)
- [Tarjeta 6 — Smart Pricing y Elasticidad Económica](#tarjeta-6)
- [Tarjeta 7 — Estacionalidad, Dashboard y Exportación](#tarjeta-7)


---
## 📋 TARJETA 1: ETL ROBUSTO Y AUDITORÍA DE CALIDAD DE DATOS
<a id='tarjeta-1'></a>

**Objetivo:** Ingestar, limpiar y auditar el dataset de forma reproducible aplicando normalización
de texto robusta (unicodedata + regex) que resuelve el patrón "Garbage In, Garbage Out" típico de
sistemas de punto de venta sin validación de entrada.

**Tratamientos clave:**
- `tarifa` y `tipo_venta`: 415/346 variantes crudas colapsadas a 5/2 categorías reales.
- `edad`: clip doble (lower=0, upper=100) para cerrar outliers en ambos extremos.
- Detección y cuantificación de corrupciones de encoding (espacios internos tipo `IN DIVIDUAL`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import unicodedata
import re
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_theme(style='whitegrid', palette='muted')

# ── DETECCIÓN DE ENTORNO ──────────────────────────────────────────────────────
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print(f"Entorno detectado: {'Google Colab' if EN_COLAB else 'Local / VS Code'}")

# ── INGESTA SEGÚN ENTORNO ─────────────────────────────────────────────────────
if EN_COLAB:
    import requests, io

    LINK_PARTE_1 = "https://drive.google.com/file/d/1VrlFnJBlJEZ8QRu-1MhghVFr3a90-L2f/view?usp=drive_link"
    LINK_PARTE_2 = "https://drive.google.com/file/d/1wovzkXYwuF9KhavMTwNya_MOS4P55u5D/view?usp=drive_link"

    def descargar_csv_drive(url_compartida):
        match = re.search(r'/d/([a-zA-Z0-9-_]+)', url_compartida)
        file_id = match.group(1)
        url_base = "https://docs.google.com/uc?export=download"
        session = requests.Session()
        response = session.get(url_base, params={'id': file_id}, stream=True)
        token = next((v for k, v in response.cookies.items() if k.startswith('download_warning')), None)
        if token:
            response = session.get(url_base, params={'id': file_id, 'confirm': token}, stream=True)
        return response.text

    print("Descargando particiones desde Google Drive...")
    df_p1 = pd.read_csv(io.StringIO(descargar_csv_drive(LINK_PARTE_1)))
    df_p2 = pd.read_csv(io.StringIO(descargar_csv_drive(LINK_PARTE_2)))
else:
    RUTA_P1 = 'usos_parques_2024_2025.csv'
    RUTA_P2 = 'usos_parques_2024_2025_2.csv'
    if not os.path.exists(RUTA_P1):
        raise FileNotFoundError(f"No se encontró '{RUTA_P1}'. Coloca los CSV junto al notebook.")
    df_p1 = pd.read_csv(RUTA_P1)
    df_p2 = pd.read_csv(RUTA_P2)
    print(f"  Archivos locales cargados: {len(df_p1):,} + {len(df_p2):,} filas.")

df_p1.columns = df_p1.columns.str.strip()
df_p2.columns = df_p2.columns.str.strip()
df_crudo = pd.concat([df_p1, df_p2], ignore_index=True)
df = df_crudo.copy()
print(f"\n  Filas brutas concatenadas: {len(df):,}")
print(f"  Columnas: {list(df.columns)}")


In [ ]:
# ── AUDITORÍA DE CALIDAD CRUDA (antes de limpiar) ───────────────────────────
print("=" * 60)
print("  AUDITORÍA DE CARDINALIDAD CRUDA (PRE-LIMPIEZA)")
print("=" * 60)
for col in ['tarifa', 'tipo_venta', 'grupo', 'genero']:
    n = df[col].nunique()
    top5 = list(df[col].value_counts().head(5).index)
    print(f"  {col:12s}: {n:>4} únicos  — muestra top-5: {top5}")

print("\n  Nulos por columna clave:")
for col in ['code_anon','fecha','fechanacimiento','valor_neto','valor_descuento','genero']:
    pct = df[col].isna().mean() * 100
    print(f"    {col:20s}: {pct:.2f}% NaN")


In [ ]:
# ── NORMALIZACIÓN ROBUSTA DE TEXTO ───────────────────────────────────────────
# Principio: unicodedata.normalize('NFD') descompone caracteres acentuados
# en su base + diacrítico; luego filtramos los diacríticos (categoría 'Mn').
# Esto resuelve: tildes (PÚBLICO→PUBLICO), casing (individual→INDIVIDUAL)
# y espacios extra/internos. No depende de librerías externas como unidecode.

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')  # quitar diacríticos
    s = s.upper()
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def colapsar_corrupcion_encoding(serie):
    # Problema: 'IN DIVIDUAL', 'INDI VIDUAL' etc. son la misma palabra con
    # espacios insertados por corrupción del sistema origen.
    # Solución: quitar todos los espacios, mapear al valor canónico más frecuente.
    sin_espacios = serie.str.replace(' ', '', regex=False)
    mapa = (
        pd.concat([sin_espacios.rename('key'), serie.rename('val')], axis=1)
        .groupby('key')['val']
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )
    return sin_espacios.map(mapa).fillna(serie)


# Aplicar a columnas de texto
for col in ['tarifa', 'tipo_venta', 'grupo', 'genero']:
    df[col] = df[col].apply(normalizar_texto)

# Segunda pasada para corrupciones de encoding en tarifa y tipo_venta
df['tarifa']     = colapsar_corrupcion_encoding(df['tarifa'])
df['tipo_venta'] = colapsar_corrupcion_encoding(df['tipo_venta'])

# ── TIPADO NUMÉRICO ───────────────────────────────────────────────────────────
df['codsede']         = pd.to_numeric(df['codsede'], errors='coerce').fillna(0).astype(int)
df['codproducto']     = pd.to_numeric(df['codproducto'], errors='coerce').fillna(0).astype(int)
df['valor_neto']      = pd.to_numeric(df['valor_neto'], errors='coerce').fillna(0)
df['valor_descuento'] = pd.to_numeric(df['valor_descuento'], errors='coerce').fillna(0)

# ── FECHAS ────────────────────────────────────────────────────────────────────
df.dropna(subset=['code_anon', 'fecha'], inplace=True)
df['fecha']           = pd.to_datetime(df['fecha'],           format='mixed', errors='coerce')
df['fechanacimiento'] = pd.to_datetime(df['fechanacimiento'], format='mixed', errors='coerce')
df.dropna(subset=['fecha'], inplace=True)
df.drop_duplicates(inplace=True)

# ── EDAD CON DOBLE CLIP ───────────────────────────────────────────────────────
# clip(lower=0): evita edades negativas (fechas de nacimiento futuras)
# clip(upper=100): elimina outliers históricos (eg. nacimiento 1895) que contaminan
# los centroides de K-Means y la distribución de edad en el modelo de churn.
max_date = df['fecha'].max()
df['edad'] = ((max_date - df['fechanacimiento']).dt.days // 365).clip(lower=0, upper=100)

# ── AUDITORÍA POST-LIMPIEZA ───────────────────────────────────────────────────
print("=" * 60)
print("  AUDITORÍA DE CARDINALIDAD POST-LIMPIEZA")
print("=" * 60)
for col in ['tarifa', 'tipo_venta', 'grupo', 'genero']:
    n = df[col].nunique()
    vals = list(df[col].value_counts().head(6).index)
    print(f"  {col:12s}: {n:>3} únicos → {vals}")

print(f"\n  Edad mín: {df['edad'].min()}, máx: {df['edad'].max()}")
print(f"  Registros edad==0 (clipeados): {(df['edad']==0).sum():,}")
print(f"  Registros edad==100 (clipeados): {(df['edad']==100).sum():,}")
print(f"\n✅ Pipeline ETL completado: {df.shape[0]:,} registros | {df.shape[1]} columnas")
print(f"   Línea de tiempo: {df['fecha'].min().date()} → {df['fecha'].max().date()}")
print(f"   Usuarios únicos: {df['code_anon'].nunique():,}")


---
## 📋 TARJETA 2: INGENIERÍA DE CARACTERÍSTICAS — MATRIZ RFM EXTENDIDA
<a id='tarjeta-2'></a>

**Objetivo:** Construir la matriz `df_user` con todas las señales comportamentales por usuario,
incluyendo señales adicionales más allá de las variables RFM clásicas.

**Features nuevas incorporadas:**
- `pct_fin_semana`: proporción de visitas en fin de semana → patrón de disponibilidad
- `n_meses_activo`: diversidad temporal → distingue usuario constante vs. puntual
- `franja_dominante`: hora preferida de visita (MAÑANA/TARDE/NOCHE)
- `categoria_edad`: ciclo de vida (NIÑO/JOVEN/ADULTO/ADULTO_MAYOR)
- `genero`: ahora sí entra al modelo vía one-hot encoding


In [ ]:
# ── VARIABLES TEMPORALES Y COMPORTAMENTALES ──────────────────────────────────
df['anio']          = df['fecha'].dt.year
df['mes']           = df['fecha'].dt.month
df['dia_semana']    = df['fecha'].dt.dayofweek          # 0=lunes, 6=domingo
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)

def discretizar_hora(h):
    if 6  <= h < 12: return 'MANANA'
    if 12 <= h < 18: return 'TARDE'
    return 'NOCHE'
df['franja_horaria'] = df['hora'].apply(discretizar_hora)

def ciclo_vida(edad):
    if pd.isna(edad): return 'DESCONOCIDO'
    if edad < 13:     return 'NINO'
    if edad < 27:     return 'JOVEN'
    if edad < 60:     return 'ADULTO'
    return 'ADULTO_MAYOR'
df['categoria_edad'] = df['edad'].apply(ciclo_vida)

# ── MATRIZ RFM EXTENDIDA POR USUARIO ─────────────────────────────────────────
df_user = df.groupby('code_anon').agg(
    recencia        = ('fecha',           lambda x: (max_date - x.max()).days),
    frecuencia      = ('fecha',           'count'),
    monetario_neto  = ('valor_neto',      'sum'),
    total_descuento = ('valor_descuento', 'sum'),
    edad            = ('edad',            'first'),
    categoria_edad  = ('categoria_edad',  'first'),
    genero          = ('genero',          'first'),
    pct_fin_semana  = ('es_fin_semana',   'mean'),   # % visitas en finde
    n_meses_activo  = ('mes',             'nunique') # meses distintos con actividad
).reset_index()

df_user['ratio_descuento'] = (
    df_user['total_descuento'] /
    (df_user['monetario_neto'] + df_user['total_descuento'] + 1e-9)
)

# Franja horaria dominante (la más frecuente para cada usuario)
franja_dom = (
    df.groupby('code_anon')['franja_horaria']
    .agg(lambda x: x.value_counts().index[0])
)
df_user = df_user.join(franja_dom, on='code_anon')
df_user.rename(columns={'franja_horaria': 'franja_dominante'}, inplace=True)

print(f"✅ Matriz RFM extendida: {len(df_user):,} usuarios únicos")
print(f"   Columnas: {list(df_user.columns)}")
print()
print("Resumen estadístico:")
print(df_user[['recencia','frecuencia','monetario_neto','edad','pct_fin_semana','n_meses_activo']].describe().round(2))

# ── NOTA CRÍTICA: naturaleza del dataset ──────────────────────────────────────
freq_dist = df_user['frecuencia'].value_counts().sort_index()
pct_1_visita = (df_user['frecuencia'] == 1).mean() * 100
print(f"\n⚠️  NOTA: {pct_1_visita:.1f}% de los usuarios registraron exactamente 1 visita.")
print("   Esto implica que churn/recencia para el 90% depende de una única fecha.")
print("   El modelo captura señal real desde edad, monetario, genero y franja horaria.")


---
## 📋 TARJETA 3: MODELADO NO SUPERVISADO — SEGMENTACIÓN K-MEANS
<a id='tarjeta-3'></a>

**Objetivo:** Segmentar la base de usuarios en 4 grupos, usando 6 features de clustering
para capturar segmentos de comportamiento más ricos.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Features de clustering: variables RFM + comportamentales
FEATURES_CLUSTER = ['recencia','frecuencia','monetario_neto','edad','pct_fin_semana','n_meses_activo']

scaler_k = StandardScaler()
X_scaled = scaler_k.fit_transform(df_user[FEATURES_CLUSTER].fillna(0))

# ── MÉTODO DEL CODO ───────────────────────────────────────────────────────────
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init='auto')
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, 11), wcss, marker='o', color='royalblue', linewidth=2)
plt.title('Método del Codo — Selección de K óptimo')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

# ── K=4 ───────────────────────────────────────────────────────────────────────
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init='auto')
df_user['cluster'] = kmeans.fit_predict(X_scaled)

print("Perfil mediano por cluster:")
print(df_user.groupby('cluster')[FEATURES_CLUSTER].median().round(2))
print()

# Etiquetado semántico basado en los perfiles observados:
# cluster 0: recencia alta, frecuencia baja, no finde → Visitantes Antiguos
# cluster 1: recencia muy alta, frecuencia baja       → Perdidos / Inactivos
# cluster 2: recencia media, frecuencia media          → Ocasionales Activos
# cluster 3: recencia baja, frecuencia alta, meses++  → VIP Recurrentes
perfil = df_user.groupby('cluster')[['recencia','frecuencia','n_meses_activo']].median()
segmento_ord = perfil.sort_values(['frecuencia','n_meses_activo'], ascending=False).index

SEGMENT_MAP = {
    segmento_ord[0]: 'Usuarios VIP Recurrentes',
    segmento_ord[1]: 'Asistentes Ocasionales Activos',
    segmento_ord[2]: 'Usuarios en Riesgo de Deserción',
    segmento_ord[3]: 'Visitantes Inactivos / Perdidos',
}
df_user['segmento'] = df_user['cluster'].map(SEGMENT_MAP)

print("Distribución de segmentos:")
print(df_user['segmento'].value_counts())

# ── VISUALIZACIÓN ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
counts = df_user['segmento'].value_counts()
colores_seg = ['#2196F3','#4CAF50','#FF5722','#9C27B0']
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=colores_seg, alpha=0.85)
for bar, val in zip(bars, counts.values[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_title('Distribución de Segmentos — K-Means K=4')
ax.set_xlabel('Número de Usuarios')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()


---
## 📋 TARJETA 4: MODELADO SUPERVISADO — PREDICCIÓN DE CHURN
<a id='tarjeta-4'></a>

**Objetivo:** Entrenar el clasificador de churn con **12 features**, incorporando correctamente
las variables categóricas vía `pd.get_dummies` (one-hot encoding real).

**Anti-leakage:** `recencia` sigue excluida del espacio de features porque la variable objetivo
`churn` se define directamente a partir de ella.

**Features utilizadas:**
- Base: `frecuencia`, `monetario_neto`, `edad`, `ratio_descuento`
- Comportamentales: `pct_fin_semana`, `n_meses_activo`
- Categóricas one-hot: `franja_dominante`, `categoria_edad`, `genero`


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler as ScalerSup

# ── DEFINICIÓN DE CHURN ───────────────────────────────────────────────────────
# Un usuario tiene churn=1 si su recencia supera la mediana de su propio cluster.
df_user['churn'] = df_user.groupby('cluster')['recencia'].transform(
    lambda x: (x > x.median()).astype(int)
)
print(f"Distribución de churn: {df_user['churn'].value_counts().to_dict()}")

# ── DIAGNÓSTICO DE LEAKAGE (matriz de correlación) ───────────────────────────
cols_corr = ['recencia','frecuencia','monetario_neto','edad','ratio_descuento',
             'pct_fin_semana','n_meses_activo','churn']
corr_matrix = df_user[cols_corr].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', fmt='.2f',
            linewidths=0.5, mask=mask, vmin=-1, vmax=1)
plt.title('Matriz de Correlación — Diagnóstico de Fuga de Datos', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"corr(recencia, churn) = {corr_matrix.loc['recencia','churn']:.4f}")
print("→ recencia EXCLUIDA del modelo por data leakage directo.")

# ── ONE-HOT ENCODING DE CATEGÓRICAS ──────────────────────────────────────────
franja_dummies = pd.get_dummies(df_user['franja_dominante'], prefix='franja', drop_first=True).astype(int)
edad_dummies   = pd.get_dummies(df_user['categoria_edad'],   prefix='edad',   drop_first=True).astype(int)
genero_dummies = pd.get_dummies(df_user['genero'],           prefix='genero', drop_first=True).astype(int)

df_user_model = pd.concat([df_user, franja_dummies, edad_dummies, genero_dummies], axis=1)

# ── ESPACIO DE FEATURES ENRIQUECIDO ──────────────────────────────────────────
FEATURES_BASE  = ['frecuencia','monetario_neto','edad','ratio_descuento',
                  'pct_fin_semana','n_meses_activo']
FEATURES_DUMMY = (list(franja_dummies.columns) +
                  list(edad_dummies.columns) +
                  list(genero_dummies.columns))
FEATURES_PRED  = FEATURES_BASE + FEATURES_DUMMY

print(f"\nFeatures del modelo ({len(FEATURES_PRED)}): {FEATURES_PRED}")

X = df_user_model[FEATURES_PRED].fillna(0)
y = df_user_model['churn']

# División estratificada 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Escalar para Logística (RF no lo requiere)
scaler_sup = ScalerSup()
X_train_s = scaler_sup.fit_transform(X_train)
X_test_s  = scaler_sup.transform(X_test)

log_reg = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
log_reg.fit(X_train_s, y_train)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

print(f"\n✅ Modelos entrenados. Train: {len(X_train):,} | Test: {len(X_test):,}")


---
## 📋 TARJETA 5: EVALUACIÓN DE MODELOS Y CURVA ROC
<a id='tarjeta-5'></a>

**Resultado:**
- Random Forest: **ROC-AUC ≈ 0.7537**
- La capacidad predictiva proviene principalmente de `monetario_neto` (feature más importante,
  53% de importancia), `edad` (24%) y `pct_fin_semana` (11%) — tres features con señal real.


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

y_pred_log = log_reg.predict(X_test_s)
y_pred_rf  = rf_clf.predict(X_test)

print("=" * 70)
print("     REPORTE TÉCNICO DE RENDIMIENTO — MODELOS DE CHURN")
print("=" * 70)

print("\n--- REGRESIÓN LOGÍSTICA ---")
print(classification_report(y_test, y_pred_log))
auc_log = roc_auc_score(y_test, log_reg.predict_proba(X_test_s)[:,1])
print(f"ROC-AUC: {auc_log:.4f}")

print("\n--- RANDOM FOREST (MODELO GANADOR) ---")
print(classification_report(y_test, y_pred_rf))
probas_rf = rf_clf.predict_proba(X_test)[:,1]
auc_rf = roc_auc_score(y_test, probas_rf)
print(f"ROC-AUC: {auc_rf:.4f}")

# ── CURVA ROC ─────────────────────────────────────────────────────────────────
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, probas_rf)
fpr_log, tpr_log, _ = roc_curve(y_test, log_reg.predict_proba(X_test_s)[:,1])

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf,  tpr_rf,  label=f'Random Forest (AUC={auc_rf:.4f})',
         color='#d62728', lw=2)
plt.plot(fpr_log, tpr_log, label=f'Reg. Logística (AUC={auc_log:.4f})',
         color='#1f77b4', lw=2, linestyle='--')
plt.plot([0,1],[0,1], color='grey', lw=1.5, linestyle=':', label='Clasificador aleatorio')
plt.fill_between(fpr_rf, tpr_rf, alpha=0.08, color='#d62728')
plt.title('Curva ROC — Validación de Generalización del Modelo', fontweight='bold')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── IMPORTANCIA DE FEATURES ───────────────────────────────────────────────────
importancias = pd.Series(rf_clf.feature_importances_, index=FEATURES_PRED).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
importancias.plot(kind='barh', color='steelblue', alpha=0.85, ax=ax)
ax.set_title('Importancia de Características — Random Forest')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

print("=" * 70)
print(f"VEREDICTO: Random Forest supera a Reg. Logística en {(auc_rf-auc_log)*100:.2f}pp de AUC.")
print(f"ROC-AUC Random Forest: {auc_rf:.4f}")
print("=" * 70)


---
## 📋 TARJETA 6: SMART PRICING — ELASTICIDAD ECONÓMICA
<a id='tarjeta-6'></a>

**Objetivo:** Cuantificar la sensibilidad al descuento y calcular el umbral óptimo real.

> El umbral calculado sobre los datos reales es **11.92%**, correspondiente al ratio de
> descuento promedio entre los usuarios que efectivamente lograron 3 o más visitas.


In [ ]:
# ── CORRELACIÓN DESCUENTO vs FRECUENCIA ──────────────────────────────────────
corr_pearson  = df_user['ratio_descuento'].corr(df_user['frecuencia'])
corr_spearman = df_user['ratio_descuento'].corr(df_user['frecuencia'], method='spearman')

print(f"Correlación Pearson  (descuento vs frecuencia): {corr_pearson:.4f}")
print(f"Correlación Spearman (descuento vs frecuencia): {corr_spearman:.4f}")
print("→ Correlación casi nula: el descuento masivo NO predice mayor frecuencia.")
print("→ Los descuentos deben aplicarse QUIRÚRGICAMENTE, no de forma masiva.")

# ── UMBRAL ÓPTIMO ─────────────────────────────────────────────────────────────
# Promedio del ratio_descuento de los usuarios que ya lograron >=3 visitas
# (umbral que no distorsionó su comportamiento, pero sí pudo haber atraído)
opt_discount = df_user[df_user['frecuencia'] >= 3]['ratio_descuento'].mean()
print(f"\nUmbral técnico de descuento para fidelización: {opt_discount:.4f} ({opt_discount:.2%})")

# ── VISUALIZACIÓN ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Smart Pricing — Análisis de Elasticidad Precio-Frecuencia', fontweight='bold')

p95_f = df_user['frecuencia'].quantile(0.95)
muestra = df_user.sample(min(5000, len(df_user)), random_state=42)
axes[0].scatter(muestra['ratio_descuento'], muestra['frecuencia'].clip(upper=p95_f),
                alpha=0.25, s=12, color='seagreen')
axes[0].axvline(opt_discount, color='red', linestyle='--', lw=2,
                label=f'Umbral óptimo: {opt_discount:.1%}')
axes[0].set_xlabel('Ratio de Descuento')
axes[0].set_ylabel('Frecuencia de visitas (P95)')
axes[0].set_title(f'Scatter: Descuento vs Frecuencia\n(Spearman ρ = {corr_spearman:.4f})')
axes[0].legend()

orden_seg = df_user.groupby('segmento')['frecuencia'].median().sort_values(ascending=False).index
sns.boxplot(data=df_user, x='segmento', y='ratio_descuento',
            order=orden_seg, palette='Set2', ax=axes[1])
axes[1].set_title('Distribución del Descuento por Segmento')
axes[1].set_xlabel('')
axes[1].set_ylabel('Ratio de Descuento')
axes[1].axhline(opt_discount, color='red', linestyle='--', lw=1.5,
                label=f'Umbral {opt_discount:.1%}')
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nRECOMENDACIÓN ESTRATÉGICA:")
print(f"  Aplicar descuentos de hasta {opt_discount:.1%} SOLO a segmentos de baja frecuencia.")
print(f"  NO aplicar a 'Usuarios VIP': son insensibles al precio.")


---
## 📋 TARJETA 7: ESTACIONALIDAD, DASHBOARD Y EXPORTACIÓN
<a id='tarjeta-7'></a>

**Objetivo:** Mapa volumétrico mensual 2024–2025 y exportación de la matriz de analítica final.


In [ ]:
# ── ESTACIONALIDAD MENSUAL 2024 vs 2025 ──────────────────────────────────────
comp = df.groupby(['anio', 'mes']).agg(
    visitas  = ('code_anon', 'count'),
    ingresos = ('valor_neto', 'sum')
).reset_index()

MESES_LBL = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
COLORES    = {2024: 'royalblue', 2025: 'seagreen'}

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle('Perfil de Estacionalidad Histórica — 2024 vs 2025', fontsize=14, fontweight='bold')

for anio, color in COLORES.items():
    sub = comp[comp['anio'] == anio]
    if not sub.empty:
        axes[0].plot(sub['mes'], sub['visitas'], marker='o', label=str(anio), color=color, lw=2)
        axes[1].bar(sub['mes'] + (0 if anio == 2024 else 0.4), sub['ingresos'],
                    width=0.4, label=str(anio), color=color, alpha=0.75)

axes[0].set_title('Visitas Mensuales')
axes[0].set_ylabel('Total Visitas')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].set_title('Ingresos Netos Mensuales')
axes[1].set_ylabel('COP')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6):,}M'))
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(MESES_LBL)
plt.tight_layout()
plt.show()

# ── RESUMEN DE HALLAZGOS ──────────────────────────────────────────────────────
for anio in [2024, 2025]:
    sub = comp[comp['anio'] == anio]
    if not sub.empty:
        idx_pico = sub['visitas'].idxmax()
        mes_pico = MESES_LBL[int(sub.loc[idx_pico,'mes'])-1]
        print(f"  Mes pico {anio}: {mes_pico} — {sub.loc[idx_pico,'visitas']:,} visitas")

# ── EXPORTACIÓN ───────────────────────────────────────────────────────────────
df_user.to_csv('df_resultados_analisis_parques.csv', index=False)
print(f"\n✅ Exportado: df_resultados_analisis_parques.csv  ({len(df_user):,} usuarios)")
print(f"   Columnas exportadas: {list(df_user.columns)}")


---
## 🔬 Conclusiones Estratégicas

### A. Calidad de Datos — Lo que el compilador no detecta

El pipeline incorpora normalización robusta con `unicodedata` que resuelve el problema
"Garbage In, Garbage Out" identificado durante la auditoría inicial de los datos:

- `tarifa`: 415 variantes crudas → **5 categorías reales** (TA, TB, TC, TD, ABIERTO AL PUBLICO)
- `tipo_venta`: 346 variantes → **2 categorías reales** (INDIVIDUAL, EMPRESARIAL)
- La limpieza de texto (aunque `tarifa`/`tipo_venta` no son features del modelo) es
  esencial para análisis descriptivos y reportes de estacionalidad correctos.

### B. Feature Engineering — 12 Features

El modelo incorpora variables comportamentales adicionales junto a las variables RFM clásicas:

| Feature | Importancia RF | Rol |
|:--------|:--------------:|:----|
| `monetario_neto` | ~53% | Gasto total: mejor predictor de engagement |
| `edad` | ~24% | Ciclo de vida: comportamiento varía con generación |
| `pct_fin_semana` | ~11% | Disponibilidad: patrón de vida |
| `ratio_descuento` | ~5% | Sensibilidad al precio |
| `genero`, `franja_dominante`, `n_meses_activo` | ~7% total | Contexto comportamental |

### C. Integridad Predictiva

- **ROC-AUC: 0.7537**
- `recencia` sigue excluida por data leakage directo (correlación directa con `churn`).
- **Limitación estructural del dataset:** 89.8% de usuarios con 1 sola visita implica que
  el churn para la mayoría es equivalente a "cuándo fue su única visita". El modelo captura
  la señal disponible, pero la definición de churn ideal requeriría datos longitudinales
  de mayor densidad.

### D. Smart Pricing

- Umbral técnico de descuento: **11.92%**
- Correlación descuento-frecuencia ≈ 0 confirma que los descuentos masivos no generan
  retención — deben ser quirúrgicos y segmentados.

---
*Pipeline validado reproduciblemente sobre los datos reales del proyecto.*
